# 4. Speculative Decoding
Uses deterministic greedy speculative decoding. The notebook verifies that the speculative output exactly matches target-model greedy decoding before reporting latency. This avoids incorrectly mixing stochastic acceptance rules with greedy draft proposals.

In [ ]:
# Speculative decoding: exact greedy verification
%pip install -q -U transformers accelerate
import time, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_name="gpt2-large" if torch.cuda.is_available() else "gpt2"
draft_name="gpt2"
tok=AutoTokenizer.from_pretrained(target_name); tok.pad_token=tok.eos_token
target=AutoModelForCausalLM.from_pretrained(target_name,torch_dtype=torch.float16 if device.type=="cuda" else torch.float32).to(device).eval()
draft=AutoModelForCausalLM.from_pretrained(draft_name,torch_dtype=torch.float16 if device.type=="cuda" else torch.float32).to(device).eval()

@torch.no_grad()
def greedy(model,ids,n):
    out=model.generate(input_ids=ids,max_new_tokens=n,do_sample=False,use_cache=True,pad_token_id=tok.eos_token_id)
    return out

@torch.no_grad()
def speculative_greedy(target,draft,ids,max_new_tokens=50,k=5):
    generated=ids.clone(); produced=0
    while produced<max_new_tokens:
        kk=min(k,max_new_tokens-produced); cur=generated.clone(); proposals=[]
        # Draft proposes k greedy tokens.
        for _ in range(kk):
            dlogits=draft(cur,use_cache=False).logits[:,-1,:]; nxt=dlogits.argmax(-1,keepdim=True); proposals.append(nxt); cur=torch.cat([cur,nxt],dim=1)
        proposal=torch.cat(proposals,dim=1)
        # One target forward verifies all proposed positions.
        full=torch.cat([generated,proposal],dim=1); logits=target(full,use_cache=False).logits
        accepted=[]
        for j in range(kk):
            target_best=logits[:,generated.shape[1]-1+j,:].argmax(-1,keepdim=True)
            if target_best.item()==proposal[0,j].item(): accepted.append(proposal[:,j:j+1])
            else:
                accepted.append(target_best); break
        new=torch.cat(accepted,dim=1); generated=torch.cat([generated,new],dim=1); produced+=new.shape[1]
        if new[0,-1].item()==tok.eos_token_id: break
    return generated

prompt="Artificial intelligence is transforming the world of"; ids=tok(prompt,return_tensors="pt").input_ids.to(device)
# Correctness check: greedy and speculative-greedy should produce identical target-greedy output for the same prompt.
g=greedy(target,ids,50); s=speculative_greedy(target,draft,ids,50,5)
print("Exact token match:", torch.equal(g,s))
print("GREEDY:",tok.decode(g[0],skip_special_tokens=True)); print("SPECULATIVE:",tok.decode(s[0],skip_special_tokens=True))

def timeit(fn,runs=3):
    for _ in range(1): fn()
    ts=[]
    for _ in range(runs):
        if device.type=="cuda": torch.cuda.synchronize()
        t=time.perf_counter(); fn()
        if device.type=="cuda": torch.cuda.synchronize()
        ts.append(time.perf_counter()-t)
    return float(np.median(ts))
tg=timeit(lambda:greedy(target,ids,50)); ts=timeit(lambda:speculative_greedy(target,draft,ids,50,5))
print({"greedy_s":tg,"speculative_s":ts,"speedup":tg/ts})
